# KBO Hitter Performance Analysis Using SQL

This project analyzes offensive performance in the Korean Baseball Organization (KBO) League using SQL and batting statistics from the 2021–2025 seasons. The analysis focuses on offensive production and advanced metrics such as OPS, wRC+, and WAR.

In [3]:
import pandas as pd
import sqlite3

df = pd.read_csv("kbo_batting_stats_by_season_1982-2025.csv")

In [4]:
df = df[df['Year'] >= 2021]

In [5]:
conn = sqlite3.connect(':memory:')
df.to_sql('batting', conn, if_exists='replace', index=False)

1473

In [6]:
df[['Name','Year','Team']].head()

,Name,Year,Team
3679,백용환,2021,KIA
3680,백용환,2021,한화
3681,백용환,2022,한화
3695,이성우,2021,LG
3746,김선빈,2021,KIA


## Exploratory Analysis

Which individual player seasons recorded the highest OPS values between 2021 and 2025?

In [8]:
query = """
SELECT Name,
       Team,
       Year,
       PA,
       ROUND(OPS,3) AS OPS
FROM batting
WHERE PA >= 100
ORDER BY OPS DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,Name,Team,Year,PA,OPS
0,나성범,KIA,2023,253,1.098
1,제러드,두산,2024,169,1.080
2,김도영,KIA,2024,625,1.067
3,구자욱,삼성,2024,568,1.044
4,디아즈,삼성,2025,628,1.025
5,안현민,KT,2025,482,1.018
6,데이비슨,NC,2024,567,1.003
7,이정후,키움,2022,627,0.996
8,양의지,NC,2021,570,0.995
9,로하스,KT,2024,670,0.989


### Findings

Na Sung-bum's 2023 season and Kim Do-young's 2024 season ranked among the most productive offensive seasons in the dataset, each posting OPS values above 1.000. Several players from different teams achieved elite offensive production during the 2021–2025 period, highlighting the high level of offensive talent across the KBO League.

## Research Question 1

Which player seasons generated the most overall value based on WAR?

In [9]:
query = """
SELECT Name,
       Team,
       Year,
       ROUND(WAR,2) AS WAR,
       ROUND(OPS,3) AS OPS
FROM batting
WHERE PA >= 100
ORDER BY WAR DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,Name,Team,Year,WAR,OPS
0,이정후,키움,2022,8.77,0.996
1,김도영,KIA,2024,8.59,1.067
2,송성문,키움,2025,8.58,0.917
3,나성범,KIA,2022,7.16,0.910
4,홍창기,LG,2021,7.14,0.864
5,강백호,KT,2021,7.02,0.971
6,홍창기,LG,2023,6.89,0.856
7,양의지,두산,2025,6.79,0.939
8,안현민,KT,2025,6.77,1.018
9,노시환,한화,2023,6.74,0.929


### Findings

Lee Jung-hoo's 2022 season generated the highest WAR in the dataset, followed closely by Kim Do-young's 2024 season and Song Sung-moon's 2025 season. While some players combined elite OPS values with high WAR totals, others generated substantial overall value despite posting more modest offensive statistics. This suggests that overall player value is not determined by offensive production alone.

## Research Question 2

Which player seasons generated the strongest offensive production based on wRC+?

In [10]:
query = """
SELECT Name,
       Team,
       Year,
       ROUND([wRC+],1) AS wRC_plus,
       ROUND(OPS,3) AS OPS
FROM batting
WHERE PA >= 100
ORDER BY [wRC+] DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,Name,Team,Year,wRC_plus,OPS
0,나성범,KIA,2023,209.8,1.098
1,이정후,키움,2022,186.4,0.996
2,제러드,두산,2024,183.7,1.080
3,안현민,KT,2025,182.7,1.018
4,김도영,KIA,2024,172.5,1.067
5,이정후,키움,2021,171.8,0.960
6,오스틴,LG,2025,171.6,0.988
7,피렐라,삼성,2022,170.8,0.976
8,강백호,KT,2021,168.7,0.971
9,양의지,NC,2021,168.5,0.995


### Findings

Na Sung-bum's 2023 season recorded the highest wRC+ in the dataset, indicating exceptional offensive production relative to league average. Several other elite seasons, including those by Lee Jung-hoo, Kim Do-young, and Kang Baek-ho, also achieved wRC+ values well above 150. These results highlight a number of historically outstanding offensive performances in the KBO League between 2021 and 2025.

## Research Question 3

Which player seasons generated high offensive production despite relatively low home run totals?

In [11]:
query = """
SELECT Name,
       Team,
       Year,
       HR,
       ROUND(OPS,3) AS OPS,
       ROUND([wRC+],1) AS wRC_plus
FROM batting
WHERE PA >= 300
  AND HR <= 10
  AND [wRC+] >= 140
ORDER BY [wRC+] DESC;
"""

pd.read_sql_query(query, conn)

,Name,Team,Year,HR,OPS,wRC_plus
0,이정후,키움,2021,7,0.960,171.8
1,홍창기,LG,2021,4,0.864,164.4
2,홍창기,LG,2023,1,0.856,161.3
3,박건우,NC,2022,10,0.866,147.3
4,이정후,키움,2023,6,0.861,146.6
5,김성윤,삼성,2025,6,0.893,146.2
6,문성주,LG,2022,6,0.823,145.6
7,박건우,두산,2021,6,0.841,143.3
8,홍창기,LG,2024,5,0.857,142.3
9,문보경,LG,2022,9,0.833,141.9


### Findings

Several players generated elite offensive production despite recording relatively few home runs. Notably, Lee Jung-hoo's 2021 season produced a wRC+ of 171.8 with only seven home runs, while Hong Chang-ki recorded wRC+ values above 160 in multiple seasons despite hitting four or fewer home runs. These results suggest that offensive value can be generated through on-base ability, contact quality, and overall consistency rather than home run power alone.

## Research Question 4

Which teams demonstrated the strongest offensive production based on average wRC+?

In [13]:
query = """
SELECT Team,
       ROUND(AVG([wRC+]),1) AS Avg_wRC_plus
FROM batting
WHERE PA >= 100
GROUP BY Team
ORDER BY Avg_wRC_plus DESC;
"""

pd.read_sql_query(query, conn)

,Team,Avg_wRC_plus
0,LG,108.0
1,두산,99.3
2,KIA,99.3
3,NC,98.0
4,SSG,96.8
5,삼성,94.2
6,롯데,93.8
7,KT,93.8
8,키움,91.3
9,한화,85.8


### Findings

LG recorded the highest average wRC+ among qualified KBO hitters during the 2021–2025 period, followed by Doosan and KIA. Doosan and KIA also ranked near the top of the league, while Hanwha recorded the lowest average wRC+ among qualified teams. These results suggest meaningful differences in offensive efficiency across KBO organizations during the period studied.


## Conclusion

This project analyzed offensive performance in the KBO League from 2021 to 2025 using SQL and advanced batting metrics. While OPS identified some of the most productive offensive seasons, WAR and wRC+ provided additional perspectives on overall player value and offensive contribution.

The analysis highlighted several elite seasons, including those by Lee Jung-hoo, Kim Do-young, and Na Sung-bum. It also showed that strong offensive performance can be achieved without relying heavily on home run power, as players such as Lee Jung-hoo and Hong Chang-ki consistently generated high offensive value through on-base ability and overall offensive efficiency.

At the team level, LG recorded the strongest average offensive production based on wRC+, while several other clubs performed near league average. Overall, the findings demonstrate that offensive performance cannot be fully explained by home run production alone and highlight the value of advanced metrics when evaluating player and team success.
